# D1 · Comparación inter-método

**Spec:** [`docs/spec_D1_v2_codex_method_comparison.md`](../docs/spec_D1_v2_codex_method_comparison.md)  |  **Bloque:** D · Método  |  **Run por defecto:** `ROXs12b_realigned`

Compara los métodos de extracción sobre 33 controles y emite `recommended_method` (nunca fija el canónico).

| | |
|---|---|
| **Entrada** | C2/C3/C4 + G1 verdicts |
| **Salida (QC/productos)** | `stages/stage_x10_qc.json` |
| **Consume aguas abajo** | D2 (consume el canónico de config) |


## Qué hace D1 y cómo decide

D1 compara los métodos de extracción **por pares y por banda** para decidir cuáles son consistentes, y emite un `recommended_method` — **nunca fija el canónico** (esa es la decisión humana).

**El estadístico es un t control-centrado** (`statistics.kind = t_control_centred`): para cada banda y par, compara la diferencia de continuo del objeto contra la **distribución de las diferencias de los 33 controles** (df = 32). Al restar `mu_ctrl` (la diferencia media de controles = sesgo_i − sesgo_j), **D1 ya está referenciado a controles** — por eso el pedestal de sobre-sustracción NO driva su veredicto (el trabajo de referenciación de C3/D2 solo puso a D2 al nivel de lo que D1 ya hacía).

**Bandas:** B1–B6 (continuo), LHa/LHb/LOI (líneas). **Umbrales:** `|t|>2.08` divergente (p<0.0455), `|t|>3.25` fuerte (p<0.0027).

**Veredicto = `divergent_continuum`** en el par primario (psffit vs optimal_psfsub, los dos validados por G1): **B6 t=+4.1 (fuerte)** y B2 t=−3.0 (marginal). Como es un t control-centrado, ese +4.1 es el **sistemático cromático genuino** (~1.35× en nivel), no el pedestal. `optimal_ls` es el **outlier**: todos sus pares divergen 18–34 → G1 lo rechaza.

`recommended_method = None` (D1 se niega a auto-elegir con divergencia); el humano eligió **psffit** ([`docs/d1_canonical_method_decision.md`](../docs/d1_canonical_method_decision.md)). La `action = iterate_C1_refine_PSF_before_PCA` queda **reconocida pero no accionada** (Psfao ya está; B6 aceptado como sistemática presupuestada).


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # o ROXs12b_B_adp para comparar
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage_x10_compare.sh --run-id $RUN
```

Moderado.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Añade notebooks/ (para _nbcommon) y la RAÍZ del repo (para importar musepipe),
# funcione el cwd en notebooks/ o en la raíz del repo.
_here = os.getcwd()
if os.path.basename(_here) != 'notebooks' and os.path.isdir(os.path.join(_here, 'notebooks')):
    _here = os.path.join(_here, 'notebooks')
for _p in (_here, os.path.dirname(_here)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
_root = str(nb.project_root())
if _root not in sys.path:
    sys.path.insert(0, _root)   # asegura 'import musepipe'
RUN_ID = nb.resolve_run_id(None)
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage_x10_compare.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc('stages/stage_x10_qc.json', RUN_ID)
nb.show(qc, keys=['verdict', 'action', 'recommended_method', 'kind', 'n_controls'], title='D1')


## Resultados que llevaron a la conclusión

Veredicto, t control-centrado del par primario por banda, y el outlier `optimal_ls`.


In [ ]:
from scipy.stats import t as t_dist
q = nb.load_qc('stages/stage_x10_qc.json', RUN_ID)
st = q['statistics']
# Umbrales t derivados de los p oficiales del QC (dependen de df=n_controls-1):
df = st['n_controls'] - 1
T_DIV = t_dist.ppf(1 - st['p_divergent'] / 2, df)
T_STR = t_dist.ppf(1 - st['p_strong'] / 2, df)
print('veredicto:', q['verdict'], '| recommended:', q['recommended_method'], '| action:', q['action'])
print(f"estadístico: {st['kind']} (n_controles={st['n_controls']}, df={df}); "
      f"umbral divergente p<{st['p_divergent']} (|t|>{T_DIV:.2f}), fuerte p<{st['p_strong']} (|t|>{T_STR:.2f})")
pp = 'psffit_vs_optimal_psfsub'
bands = list(q['t_matrix'][pp].keys())
print(f'\nt control-centrado del par primario ({pp}):')
for b in bands:
    t = q['t_matrix'][pp][b]
    flag = '  <-- FUERTE' if abs(t) > T_STR else ('  <- marginal' if abs(t) > T_DIV else '')
    print(f'   {b:4s}: t = {t:+.2f}{flag}')
print('\noptimal_ls es el outlier (|t| máx por par):')
for pair, row in q['t_matrix'].items():
    if 'optimal_ls' in pair:
        tmax = max(abs(v) for v in row.values())
        print(f'   {pair:34s} |t|max = {tmax:.1f}')


## Plot 1 — el t control-centrado por par × banda

Del `t_matrix` del QC. Rojo/azul = divergencia (|t| grande). **psffit vs optimal_psfsub** (fila primaria) es consistente salvo **B6 (+4.1)** y B2 (−3.0); **todos los pares con `optimal_ls`** divergen 18–34 (sobre-sustracción) → ls rechazado. psffit vs aperture es consistente en todo.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.stats import t as t_dist
    q = nb.load_qc('stages/stage_x10_qc.json', RUN_ID)
    st = q['statistics']
    T_STR = t_dist.ppf(1 - st['p_strong'] / 2, st['n_controls'] - 1)
    tm = q['t_matrix']
    pairs = list(tm.keys())
    bands = list(tm[pairs[0]].keys())
    M = np.array([[tm[p].get(b, np.nan) for b in bands] for p in pairs])
    fig, ax = plt.subplots(figsize=(9, 4.2))
    im = ax.imshow(M, cmap='RdBu_r', vmin=-5, vmax=5, aspect='auto')
    ax.set_xticks(range(len(bands))); ax.set_xticklabels(bands)
    ax.set_yticks(range(len(pairs))); ax.set_yticklabels([p.replace('_vs_', ' vs ') for p in pairs], fontsize=8)
    for i in range(len(pairs)):
        for j in range(len(bands)):
            v = M[i, j]
            if np.isfinite(v):
                ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=7, color='k' if abs(v) < 3 else 'w')
    ax.set_title(f'D1 · t control-centrado por par × banda (|t|>{T_STR:.2f} fuerte)')
    fig.colorbar(im, label='t'); fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'd1_compare'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'tmatrix.png', dpi=110); print('figura ->', outdir / 'tmatrix.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — el par primario: qué driva `divergent_continuum`

El t del par primario (psffit vs optimal_psfsub) por banda, con los umbrales divergente (2.08) y fuerte (3.25). **B6 (+4.1) cruza el umbral fuerte** y B2 (−3.0) el divergente → veredicto `divergent_continuum`. Es el sistemático cromático genuino (~1.35×), ya libre del pedestal (t control-centrado).


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.stats import t as t_dist
    q = nb.load_qc('stages/stage_x10_qc.json', RUN_ID)
    st = q['statistics']; df = st['n_controls'] - 1
    t_div = t_dist.ppf(1 - st['p_divergent'] / 2, df)   # umbrales oficiales (p del QC)
    t_str = t_dist.ppf(1 - st['p_strong'] / 2, df)
    row = q['t_matrix']['psffit_vs_optimal_psfsub']
    bands = list(row.keys()); tv = [row[b] for b in bands]
    cols = ['tab:red' if abs(v) > t_str else ('tab:orange' if abs(v) > t_div else '0.6') for v in tv]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(bands, tv, color=cols)
    for s in (t_div, t_str):
        ax.axhline(s, color='k', ls=':', lw=0.8); ax.axhline(-s, color='k', ls=':', lw=0.8)
    ax.axhline(0, color='k', lw=0.6)
    ax.set_ylabel('t control-centrado')
    ax.set_title('D1 · par primario psffit vs optimal_psfsub → divergent_continuum (B6 fuerte, B2 marginal)')
    fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'd1_compare'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'primary_pair.png', dpi=110); print('figura ->', outdir / 'primary_pair.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **Decisión humana: canónico = `psffit`** (validado por G1, físico en el borde); D1 solo recomienda (`recommended_method=None` con divergencia). · [`docs/d1_canonical_method_decision.md`](../docs/d1_canonical_method_decision.md)
- **Veredicto = `divergent_continuum`** en el par primario: B6 t=+4.1 (fuerte), B2 t=−3.0 (marginal). t **control-centrado** → es el sistemático genuino (~1.35×), no el pedestal.
- **`optimal_ls` rechazado**: outlier en todos sus pares (|t| 18–34) por sobre-sustracción.
- **B6 aceptado como sistemática presupuestada**: `action=iterate_C1` reconocida pero NO accionada (Psfao ya está; ver D2 y la referenciación de continuo). · [`docs/d2_red_continuum_diagnosis.md`](../docs/d2_red_continuum_diagnosis.md)


## Conclusión (registrada)

**D1: veredicto `divergent_continuum` (par primario psffit vs optimal_psfsub); t control-centrado; recommended_method=None.**

- **Fecha:** D1 v2 sobre el run realineado (2026-07-09).
- **Estadístico:** t control-centrado (33 controles, df=32) → ya libre del pedestal de sobre-sustracción.
- **Driver:** B6 t=+4.1 (fuerte), B2 t=−3.0 (marginal) = sistemático cromático genuino (~1.35× en nivel).
- **ls rechazado:** outlier en todos sus pares (|t| 18–34).
- **Canónico:** el humano eligió **psffit** (D1 no auto-elige con divergencia).
- **Acción:** `iterate_C1` reconocida pero no accionada (B6 aceptado como sistemática presupuestada; ver D2).
